In [ ]:
# Preprocessing steps are saved into this notebook

In [ ]:
# imports
import pandas as pd
import os
import warnings
#import datetime
#import json
import numpy as np
#from pprint import pprint
from sklearn.preprocessing import MinMaxScaler
import joblib

In [2]:
# we can have this step to ensure the correct formatting, but maybe not necessary
#pd.set_option('display.float_format',lambda x: "%.3f" % x)
warnings.filterwarnings('ignore')
# some helpers
def describe_numeric_col(x):
    """
    Parameters:
        x (pd.Series): Pandas col to describe.
    Output:
        y (pd.Series): Pandas series with descriptive stats. 
    """
    return pd.Series(
        [x.count(), x.isnull().count(), x.mean(), x.min(), x.max()],
        index=["Count", "Missing", "Mean", "Min", "Max"]
    )

def impute_missing_values(x, method="mean"):
    """
    Parameters:
        x (pd.Series): Pandas col to describe.
        method (str): Values: "mean", "median"
    """
    if (x.dtype == "float64") | (x.dtype == "int64"):
        x = x.fillna(x.mean()) if method=="mean" else x.fillna(x.median())
    else:
        x = x.fillna(x.mode()[0])
    return x

In [6]:
# to make sure the artifacts directory exists
os.makedirs("artifacts",exist_ok=True)

# load the raw data
print("Loading training data")
data = pd.read_csv("./artifacts/raw_data.csv")
#print("Total rows:", data.count())
#display(data.head(5))

Loading training data


In [7]:
# Time limit the data
max_date = "2024-01-31"
min_date = "2024-01-01"
if not max_date:
    max_date = pd.to_datetime(datetime.datetime.now().date()).date()
else:
    max_date = pd.to_datetime(max_date).date()

min_date = pd.to_datetime(min_date).date()

data["date_part"] = pd.to_datetime(data["date_part"]).dt.date
data = data[(data["date_part"] >= min_date) & (data["date_part"] <= max_date)]

min_date = data["date_part"].min()
max_date = data["date_part"].max()
date_limits = {"min_date": str(min_date), "max_date": str(max_date)}
with open("./artifacts/date_limits.json", "w") as f:
    json.dump(date_limits, f)

In [8]:
# remove irrelevant columns
data = data.drop(
    [
        "is_active", "marketing_consent", "first_booking", "existing_customer", "last_seen"
    ],
    axis=1
)

#Removing columns that will be added back after the EDA
#no_eda_split = ["domain", "country", "visited_learn_more_before_booking", "visited_faq"] # save the data to add back after EDA
#data = data.drop(
#    ["domain", "country", "visited_learn_more_before_booking", "visited_faq"],
#    axis=1
#)

In [9]:
# data cleaning
data["lead_indicator"].replace("", np.nan, inplace=True)
data["lead_id"].replace("", np.nan, inplace=True)
data["customer_code"].replace("", np.nan, inplace=True)
data = data.dropna(axis=0, subset=["lead_indicator"])
data = data.dropna(axis=0, subset=["lead_id"])

#data = data[data.source == "signup"] # ??? why ???
#result=data.lead_indicator.value_counts(normalize = True)

#print("Target value counter")
#for val, n in zip(result.index, result):
#    print(val, ": ", n)
#data

In [10]:
# create categorical data columns
vars = [
    "lead_id", "lead_indicator", "customer_group", "onboarding", "source", "customer_code"
]

for col in vars:
    data[col] = data[col].astype("object")
    print(f"Changed {col} to object type")

Changed lead_id to object type
Changed lead_indicator to object type
Changed customer_group to object type
Changed onboarding to object type
Changed source to object type
Changed customer_code to object type


In [11]:
# separate categorical and continuous columns
cont_vars = data.loc[:, ((data.dtypes=="float64")|(data.dtypes=="int64"))]
cat_vars = data.loc[:, (data.dtypes=="object")]

#print("\nContinuous columns: \n")
#pprint(list(cont_vars.columns), indent=4)
#print("\n Categorical columns: \n")
#pprint(list(cat_vars.columns), indent=4)

In [12]:
# deal with outliers
cont_vars = cont_vars.apply(lambda x: x.clip(lower = (x.mean()-2*x.std()),
                                             upper = (x.mean()+2*x.std())))
outlier_summary = cont_vars.apply(describe_numeric_col).T
outlier_summary.to_csv('./artifacts/outlier_summary.csv')

# impute the data
cat_missing_impute = cat_vars.mode(numeric_only=False, dropna=True)
cat_missing_impute.to_csv("./artifacts/cat_missing_impute.csv")

# continuous variables missing values
cont_vars = cont_vars.apply(impute_missing_values)

# categorical variables missing values
cat_vars.loc[cat_vars['customer_code'].isna(),'customer_code'] = 'None'
cat_vars = cat_vars.apply(impute_missing_values)
cat_vars.apply(lambda x: pd.Series([x.count(), x.isnull().sum()], index = ['Count', 'Missing'])).T


,Count,Missing
lead_id,11753,0
lead_indicator,11753,0
date_part,11753,0
source,11753,0
domain,11753,0
country,11753,0
customer_group,11753,0
onboarding,11753,0
customer_code,11753,0


In [13]:
# data standardisation
scaler_path = "./artifacts/scaler.pkl"

scaler = MinMaxScaler()
scaler.fit(cont_vars)

joblib.dump(value=scaler, filename=scaler_path)
print("Saved scaler in artifacts")

cont_vars = pd.DataFrame(scaler.transform(cont_vars), columns=cont_vars.columns)
#cont_vars

Saved scaler in artifacts


In [14]:
# combine data
cont_vars = cont_vars.reset_index(drop=True)
cat_vars = cat_vars.reset_index(drop=True)
data = pd.concat([cat_vars, cont_vars], axis=1)
#print(f"Data cleansed and combined.\nRows: {len(data)}")

In [15]:
# create binary column of "source"
data['bin_source'] = data['source']
values_list = ['li', 'organic','signup','fb']
data.loc[~data['source'].isin(values_list),'bin_source'] = 'Others'
mapping = {'li' : 'socials', 
           'fb' : 'socials', 
           'organic': 'group1', 
           'signup': 'group1'
           }

data['bin_source'] = data['source'].map(mapping)

In [16]:
# data columns drift
data_columns = list(data.columns)
with open('./artifacts/columns_drift.json','w+') as f:           
    json.dump(data_columns,f)

In [17]:
# save the cleaned data
data.to_csv('./artifacts/train_data_gold.csv', index=False)